**The Scenario:** You act as a university data analyst. The legacy admissions portal crashed during a system migration, and the backup applicant data is a mess. You must clean the "Applicant Records" dataset.
- Handling Missing Data: The dataset is riddled with NaNs. You must use .dropna() to remove rows completely missing an Application ID. You will use .fillna() to replace missing Intended Major and High School State values with "Unknown" that were lost during the export.


In [23]:
import pandas as pd

df = pd.read_csv("admissions_messy.csv")
df.info()
df.head()

df = df.dropna(subset=["Application ID"])
df["Intended Major"] = df["Intended Major"].fillna("Unknown")
df["High School State"] = df["High School State"].fillna("Unknown")
print(df[["Intended Major", "High School State"]])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Application ID     943 non-null    float64
 1   Applicant Name     1050 non-null   object 
 2   Email              1050 non-null   object 
 3   Intended Major     898 non-null    object 
 4   High School State  874 non-null    object 
 5   Application Fee    1050 non-null   object 
 6   Submission Date    1050 non-null   object 
dtypes: float64(1), object(6)
memory usage: 57.6+ KB
        Intended Major High School State
0                 Math                NV
1           Journalism           Unknown
2                 Math           Unknown
3          Agriculture                AZ
4              Physics                NV
...                ...               ...
1044  Computer Science                KS
1045           Unknown                AZ
1046        Journalism           Unknown

- Handling Duplicates: A server glitch caused applicants to accidentally submit identical applications twice. You must use .duplicated() to find identical application rows and .drop_duplicates(keep='first') to remove the errors and keep only the original submission.

In [24]:

duplicates = df[df.duplicated()]
print(f"Duplicate rows:\n {duplicates}")

df_drop_first = df.drop_duplicates(keep='first')
print(f"is df_drop_first:\n{df_drop_first}")


Duplicate rows:
       Application ID       Applicant Name                  Email  \
77            1210.0        Bruce Allen        ballen@skynet.com   
144           1986.0          ALICE Doe           adoe@stark.com   
147           1319.0         peter DOE              pdoe@edu.org   
160           1513.0           ARTHUR Doe         adoe@gmail.com   
177           1059.0      Victor CONNOR        vconnor@stark.com   
183           1136.0         Clark PRINCE      cprince@gmail.com   
211           1636.0         BOB Smith          bsmith@stark.com   
301           1439.0          SARA PRINCE      sprince@yahoo.com   
348           1626.0        VICTOR JONES       vjones@skynet.com   
365           1096.0       ALICE BROWN          abrown@stark.com   
414           1811.0         Diana Jones      djones@outlook.com   
437           1902.0           John JANE       jjane@outlook.com   
451           1737.0       Bruce curry          bcurry@gmail.com   
461           1601.0        hal

**Data Type Conversions:** 
- Application Fee is imported as a string (e.g., "$150.50"). You must remove the symbols and use pd.to_numeric() to turn it into a float.


In [26]:
df["Application Fee"] = (df["Application Fee"].replace(r"[\$,]", "", regex=True))
df["Application Fee"] = pd.to_numeric(df["Application Fee"], errors="coerce")

print(df["Application Fee"])

0        297.91
1       1939.68
2        277.90
3        370.25
4         53.93
         ...   
1044    1582.76
1045     862.30
1046     389.22
1047    1746.27
1048     689.05
Name: Application Fee, Length: 943, dtype: float64


- Replace “pending” and “unknown” text entries in the dates with the previous valid datetime (forward filling).


In [27]:
df["Submission Date"] = df["Submission Date"].replace(["pending", "unknown"], pd.NA)
df["Submission Date"] = df["Submission Date"].fillna(method="ffill")
print(df["Submission Date"])

0               <NA>
1         08/14/2023
2         2023/06/06
3         2023-02-24
4       Oct 16, 2023
            ...     
1044      2023/05/27
1045      10-18-2023
1046      10-20-2023
1047    Aug 17, 2023
1048      2023-06-06
Name: Submission Date, Length: 943, dtype: object


C:\Users\Dell\AppData\Local\Temp\ipykernel_15440\2064776940.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Submission Date"] = df["Submission Date"].fillna(method="ffill")


- Convert Submission Date from messy text into a proper datetime object using pd.to_datetime().

In [31]:
df["Submission Date"] = pd.to_datetime(df["Submission Date"], errors="coerce")
print(df["Submission Date"])

0             NaT
1      2023-08-14
2             NaT
3             NaT
4             NaT
          ...    
1044          NaT
1045          NaT
1046          NaT
1047          NaT
1048          NaT
Name: Submission Date, Length: 943, dtype: datetime64[ns]


- Convert the Submission Date format from various mixed formats into a uniform "M/D/Y" format.

In [34]:
df["Submission Date"] = pd.to_datetime(df["Submission Date"], errors="coerce")
df["Submission Date"] = df["Submission Date"].dt.strftime("%m/%d/%Y")
print(df["Submission Date"])

print(df[["Application Fee", "Submission Date"]])


0              NaN
1       08/14/2023
2              NaN
3              NaN
4              NaN
           ...    
1044           NaN
1045           NaN
1046           NaN
1047           NaN
1048           NaN
Name: Submission Date, Length: 943, dtype: object
      Application Fee Submission Date
0              297.91             NaN
1             1939.68      08/14/2023
2              277.90             NaN
3              370.25             NaN
4               53.93             NaN
...               ...             ...
1044          1582.76             NaN
1045           862.30             NaN
1046           389.22             NaN
1047          1746.27             NaN
1048           689.05             NaN

[943 rows x 2 columns]


 **String Manipulation:** The Applicant Name column has chaotic capitalization and extra spaces. You will use .str.title() and .str.strip() to fix this. You must also extract the email domain (e.g., extracting "yahoo.com" from "student@yahoo.com") using .str.split() to see which email providers are most common.

In [35]:
df["Applicant Name"] = (df["Applicant Name"].str.strip().str.title())
df["Applicant Name"] = (df["Applicant Name"].str.strip().str.replace(r"\s+", " ", regex=True).str.title())
print(df["Applicant Name"])

df["Email Domain"] = df["Email"].str.split("@").str[-1]
domain_counts = df["Email Domain"].value_counts()

print(domain_counts.head())

top_provider = df["Email Domain"].mode()[0]
print("Most common email provider:", top_provider)

df.to_csv("cleaned_applicant_records.csv", index=False)

0            Wanda Doe
1          Peter Jones
2       Wanda Maximoff
3         Peter Parker
4         Bruce Rogers
             ...      
1044        John Stone
1045    Diana Maximoff
1046       Steve Allen
1047    Charlie Rogers
1048       Peter Wayne
Name: Applicant Name, Length: 943, dtype: object
Email Domain
gmail.com         155
stark.com         148
dailybugle.com    134
outlook.com       131
yahoo.com         128
Name: count, dtype: int64
Most common email provider: gmail.com
